**Author:** Amanda Baright

**Date:** 04.28.2026

**Purpose:** ST 554 Final Project

# ST 554 Final Project

This final project will assess my ability to use spark to handle streaming data, use spark for fitting a machine learning model, and a few other things. The project will be split into three main components: Fitting a machine learning model using `pyspark`'s `MLlib` module, reading in a stream of data that is produced ourselves using a `.py` file, and using the model to do predictions on the stream and write the predictions out to the console.

The data is modified from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city), where the study was about relating power consumption from different time zones of Tetouan city to various factors like time of day, temperatrue, and humidity. We will treat the `Power_Zone_3` variable as our response variable, and will use all of the other varialbes as predictors. This will allow us to predict the value of `Power_Zone_3` appropriately if that reading goes offline in the future.

We are provided a chunk of the modified data to build our model on, and will then "stream data" to a folder in the form of CSV files. We will monitor this folder, and will use the fitted model to predict on the incoming data.

## Fitting Your Model

This first part will focus on model fitting. We will use the `power_ml_data.csv` which is available at: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. We first want to read in this data using a standard pandas data frame using `pd.read_csv()`. We will also use this first step to read in all of the libraries we may need. We will also initialize a Spark session and convert this pandas data frame into a spark data frame.

In [4]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (SQLTransformer, Binarizer, OneHotEncoder, 
                                VectorAssembler, PCA, StandardScaler)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [14]:
spark = SparkSession.builder.getOrCreate()

# Read data into pandas
pandas_df = pd.read_csv("power_ml_data.csv")

# Convert to spark data frame
power_df = spark.createDataFrame(pandas_df)

# Look at first few rows
power_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We then want to start building a `MLlib` Pipeline that includes all of the pre-specified transformations.

The first step will be to use a SQL transformer to rename `Power_Zone_3` as `label` and cast the `Hour` column to be stored as a `DoubleType` variable.

In [6]:
sqlTrans = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label, CAST(Hour AS DOUBLE) AS HourDouble FROM __THIS__"
)

We then want to add a component that binarizes the `Hour` column based on the column being less than 6.5 or not, which essentially breaks the `Hour` column into night vs day.

In [7]:
binarizer = Binarizer(threshold=6.5, inputCol="HourDouble", outputCol="binHour")

We then want to do one-hot encoding for the `Month` column, where each unique month in the `Month` column is transformed into its own binary column with 1 indicating the presence of that month category and 0 indicates the absence.

In [8]:
encoder = OneHotEncoder(inputCols=["Month"], outputCols=["Month_vec"])

We now want to use `VectorAssembler()` to assemble the weather related variables (`Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`). Additionally, we will want to standardize the weather variables before doing the PCA fit, as PCA is sensitive to the scale of the data.

In [9]:
weather_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="weather_features"
)

scaler_pca = StandardScaler(inputCol="weather_features", outputCol="scaled_weather", 
                            withStd=True, withMean=True)

Now that we standardized our weather features, we can run a PCA fit using two PCs in the transformation.

In [10]:
pca = PCA(k=2, inputCol="scaled_weather", outputCol="pca_features")

We then do a final feature assembly where we include the `pca_features` that were selected with the PCA fit, the binary `Hour` variable, `Power_Zone_1`, `Power_Zone_2`, and the `Month` indicator variable. These will be stored into a `unscaled_features` column by using the `VectorAssembler()` method again. We then want to standardize the final features for the Elastic Net Model, as the Elastic Net model adds a penalty term to the magnitude of the coefficients so features with larger scales will have unfairly small coefficients which may lead to some predictors to be ignored.

In [11]:
final_assembler = VectorAssembler(
    inputCols=["pca_features", "binHour", "Power_Zone_1", "Power_Zone_2", "Month_vec"],
    outputCol="unscaled_features"
)

scaler_final = StandardScaler(inputCol="unscaled_features", outputCol="features", 
                              withStd=True, withMean=True)

Now we can finally define the pipeline with all of these amazing transformations!

In [12]:
pipeline = Pipeline(stages=[
    sqlTrans, 
    binarizer, 
    encoder, 
    weather_assembler, 
    scaler_pca, 
    pca, 
    final_assembler, 
    scaler_final
])

We then want to use the `CrossValidator()` function with the `LinearRegression()` function to fit an elastic net model. We will use a pretty extensive grid with the `ParamGridBuilder()` function to examine all combinations of a few parameters for `regParam` and `elasticNetParam`. We will also fit the model using 5-fold CR with Root Mean Square Error (`rmse`) as our criterion.

In [13]:
# Set up the LinearRegression() function
lr = LinearRegression(featuresCol="features", labelCol="label")

# Define the extensive grid selection
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Set up 5-fold CV with rmse as our metric of choice.
cv = CrossValidator(estimator=lr,
                    estimatorParamMaps=paramGrid,
                    evaluator=RegressionEvaluator(metricName="rmse"),
                    numFolds=5)

Now we can finally fit the entire workflow with all of our components built above. The first step is to fit the pipeline to the data.

In [15]:
pipelineModel = pipeline.fit(power_df)

Next we will transform our data using our pipeline to get the `features` and `label` columns.

In [16]:
transformedData = pipelineModel.transform(power_df)

We can now fit the `CrossValidator()` on the transformed data.

In [17]:
cvModel = cv.fit(transformedData)

26/04/28 21:09:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/28 21:09:30 WARN Instrumentation: [33349421] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 21:09:31 WARN Instrumentation: [33349421] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 21:09:32 WARN Instrumentation: [117ecc2e] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 21:09:32 WARN Instrumentation: [117ecc2e] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 21:09:33 WARN Instrumentation: [b2f85dfb] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 21:09:33 WARN Instrumentation: [b2f85dfb] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

We then want to look at the best model and report the optimal values chosen for the tuning parameters and the CV error. Here we can extract the best model with the `.bestModel` attribute. We can then print out the `regParam` and `elasticNetParam` with their respective `.getParam()` method and find the average RMSE from the Cross Validation run to get the CV error.

In [20]:
bestModel = cvModel.bestModel
print(f"Optimal regParam: {bestModel.getRegParam()}")
print(f"Optimal elasticNetParam: {bestModel.getElasticNetParam()}")
print(f"CV RMSE: {min(cvModel.avgMetrics)}")

Optimal regParam: 0.75
Optimal elasticNetParam: 0.25
CV RMSE: 2175.2638167342766
